# Neural Network Pipeline For Monthly Equities Returns

This notebook builds a readable, time-aware feed-forward neural network pipeline in R to predict `R1M_Usd`, the one-month forward return.

The workflow is deliberately designed to avoid look-ahead bias:

1. Split the raw panel into a strict training sample and a strict holdout sample, with all observations on or after **2014-01-15** reserved for holdout.
2. Inside the pre-2014 training sample, create **5-year rolling training windows** and **retrain yearly** on the previous 5 years of data.
3. Use a **small hyperparameter grid** on a subset of pre-holdout rolling windows for model selection.
4. Refit the final model on the **last 5 pre-holdout years (2009-2013)** and score the untouched holdout set.

I am using `nnet`, which gives us a lightweight multilayer perceptron baseline directly in R. That keeps the notebook easier to run than a `keras` or `tensorflow` stack while still giving us a proper neural-network workflow.


In [12]:
# Load packages up front so notebook failures are explicit and easy to debug.
required_packages <- c("tidyverse", "lubridate", "nnet")
missing_packages <- required_packages[!vapply(required_packages, requireNamespace, logical(1), quietly = TRUE)]

if (length(missing_packages) > 0) {
  stop(
    "Install the missing packages before running this notebook: ",
    paste(missing_packages, collapse = ", ")
  )
}

invisible(lapply(required_packages, library, character.only = TRUE))

# Reproducibility matters when comparing model configurations.
set.seed(423)
options(dplyr.summarise.inform = FALSE)


In [13]:
# Load the monthly equity panel and apply the same date filter used in the Section 6 starter code.
load("data_ml.RData")

holdout_start <- as.Date("2014-01-15")
target_col <- "R1M_Usd"
feature_cols <- names(data_ml)[3:95]

data_ml <- data_ml %>%
  mutate(date = as.Date(date)) %>%
  filter(
    date > as.Date("1999-12-31"),
    date < as.Date("2019-01-01")
  ) %>%
  arrange(date, stock_id)

training_sample <- data_ml %>% filter(date < holdout_start)
holdout_sample <- data_ml %>% filter(date >= holdout_start)

split_summary <- tibble(
  sample = c("training", "holdout"),
  start_date = c(min(training_sample$date), min(holdout_sample$date)),
  end_date = c(max(training_sample$date), max(holdout_sample$date)),
  rows = c(nrow(training_sample), nrow(holdout_sample)),
  distinct_months = c(n_distinct(training_sample$date), n_distinct(holdout_sample$date))
)

split_summary


sample,start_date,end_date,rows,distinct_months
<chr>,<date>,<date>,<int>,<int>
training,2000-01-31,2013-12-31,198128,168
holdout,2014-01-31,2018-12-31,70208,60


## Helper Functions

The helpers below do four things:

- standardize features using **training-window statistics only**;
- define the rolling-window schedule;
- fit a simple neural-network regressor for each window; and
- summarize predictions with both error metrics and cross-sectional information coefficients.


In [14]:
# Small logger so long notebook runs show useful progress updates.
log_progress <- function(...) {
  timestamp <- format(Sys.time(), "%Y-%m-%d %H:%M:%S")
  cat(sprintf("[%s] %s
", timestamp, paste(..., collapse = "")))
  flush.console()
}

# Safe correlation helper for months where one vector could have zero variance.
safe_cor <- function(x, y, method = "pearson") {
  if (length(x) < 2 || sd(x) == 0 || sd(y) == 0) {
    return(NA_real_)
  }

  cor(x, y, method = method)
}

# Fit the scaler on the training window only.
make_scaler <- function(train_df, feature_cols) {
  train_matrix <- train_df %>%
    dplyr::select(all_of(feature_cols)) %>%
    as.matrix()

  center <- colMeans(train_matrix)
  scale <- apply(train_matrix, 2, sd)
  scale[is.na(scale) | scale == 0] <- 1

  list(center = center, scale = scale)
}

# Reuse the training-window scaler when transforming validation or holdout data.
apply_scaler <- function(df, feature_cols, scaler) {
  df %>%
    dplyr::select(all_of(feature_cols)) %>%
    as.matrix() %>%
    scale(center = scaler$center, scale = scaler$scale)
}

# A rolling window always trains on the previous 5 calendar years and scores the next calendar year.
make_rolling_windows <- function(score_years) {
  tibble(score_year = score_years) %>%
    mutate(
      train_start = as.Date(sprintf("%d-01-01", score_year - 5L)),
      train_end = as.Date(sprintf("%d-12-31", score_year - 1L)),
      score_start = as.Date(sprintf("%d-01-01", score_year)),
      score_end = as.Date(sprintf("%d-12-31", score_year))
    )
}

# Train on one 5-year window, then score the following period.
fit_and_score_window <- function(train_df,
                                 score_df,
                                 feature_cols,
                                 target_col,
                                 size,
                                 decay,
                                 maxit,
                                 rang,
                                 trace = FALSE,
                                 seed_offset = 0L,
                                 model_label = "window") {
  stopifnot(max(train_df$date) < min(score_df$date))

  log_progress(
    sprintf(
      "Starting %s | train rows=%s (%s to %s) | score rows=%s (%s to %s) | size=%s decay=%.4g maxit=%s rang=%.3f",
      model_label,
      format(nrow(train_df), big.mark = ","),
      min(train_df$date),
      max(train_df$date),
      format(nrow(score_df), big.mark = ","),
      min(score_df$date),
      max(score_df$date),
      size,
      decay,
      maxit,
      rang
    )
  )

  scaler <- make_scaler(train_df, feature_cols)
  x_train <- apply_scaler(train_df, feature_cols, scaler)
  y_train <- train_df[[target_col]]
  x_score <- apply_scaler(score_df, feature_cols, scaler)

  # nnet needs a sufficiently large MaxNWts when the feature count is high.
  max_weights <- (ncol(x_train) + 1L) * as.integer(size) + (as.integer(size) + 1L)

  set.seed(423L + as.integer(seed_offset))

  model <- nnet::nnet(
    x = x_train,
    y = y_train,
    size = as.integer(size),
    decay = decay,
    maxit = as.integer(maxit),
    rang = rang,
    linout = TRUE,
    skip = FALSE,
    trace = trace,
    MaxNWts = max_weights + 100L
  )

  preds <- as.numeric(predict(model, x_score, type = "raw"))

  log_progress(sprintf("Finished %s", model_label))

  preds
}

# Walk forward through a set of yearly windows.
run_walk_forward <- function(data_source,
                             windows,
                             feature_cols,
                             target_col,
                             params,
                             trace = FALSE,
                             run_label = "walk-forward") {
  log_progress(
    sprintf(
      "Running %s across %s yearly windows",
      run_label,
      nrow(windows)
    )
  )

  purrr::pmap_dfr(
    windows,
    function(score_year, train_start, train_end, score_start, score_end) {
      train_window <- data_source %>%
        filter(date >= train_start, date <= train_end)

      score_window <- data_source %>%
        filter(date >= score_start, date <= score_end)

      window_label <- sprintf(
        "%s | score_year=%s",
        run_label,
        score_year
      )

      preds <- fit_and_score_window(
        train_df = train_window,
        score_df = score_window,
        feature_cols = feature_cols,
        target_col = target_col,
        size = params$size,
        decay = params$decay,
        maxit = params$maxit,
        rang = params$rang,
        trace = trace,
        seed_offset = score_year,
        model_label = window_label
      )

      window_rank_ic <- safe_cor(preds, score_window[[target_col]], method = "spearman")
      window_hit_ratio <- mean(preds * score_window[[target_col]] > 0)

      log_progress(
        sprintf(
          "Completed %s | rank IC=%.4f | hit ratio=%.4f",
          window_label,
          window_rank_ic,
          window_hit_ratio
        )
      )

      score_window %>%
        transmute(
          stock_id,
          date,
          actual_return = .data[[target_col]],
          predicted_return = preds,
          score_year = score_year,
          train_start = train_start,
          train_end = train_end
        )
    }
  )
}

# Summarize predictions both overall and month by month.
summarise_predictions <- function(prediction_df) {
  monthly_metrics <- prediction_df %>%
    group_by(date) %>%
    summarise(
      names_scored = n(),
      monthly_ic = safe_cor(predicted_return, actual_return),
      monthly_rank_ic = safe_cor(predicted_return, actual_return, method = "spearman"),
      monthly_hit_ratio = mean(predicted_return * actual_return > 0)
    )

  overall_metrics <- prediction_df %>%
    summarise(
      observations = n(),
      rmse = sqrt(mean((predicted_return - actual_return)^2)),
      mae = mean(abs(predicted_return - actual_return)),
      overall_ic = safe_cor(predicted_return, actual_return),
      overall_rank_ic = safe_cor(predicted_return, actual_return, method = "spearman"),
      hit_ratio = mean(predicted_return * actual_return > 0)
    ) %>%
    bind_cols(
      monthly_metrics %>%
        summarise(
          mean_monthly_ic = mean(monthly_ic, na.rm = TRUE),
          mean_monthly_rank_ic = mean(monthly_rank_ic, na.rm = TRUE),
          mean_monthly_hit_ratio = mean(monthly_hit_ratio, na.rm = TRUE)
        )
    )

  list(overall = overall_metrics, monthly = monthly_metrics)
}


## Rolling Windows Inside The Training Sample

Because everything from **2014-01-15 onward** is a strict holdout set, the yearly rolling retraining is done **only inside the pre-2014 training sample**.

That gives us a realistic development workflow:

- train on years `t-5` through `t-1`;
- score year `t`;
- roll forward by one year and repeat.


In [15]:
training_years <- sort(unique(lubridate::year(training_sample$date)))
development_windows <- make_rolling_windows(score_years = training_years[training_years >= 2005])

# Use only the last few pre-holdout windows for a lightweight hyperparameter search.
tuning_windows <- development_windows %>% filter(score_year %in% 2005:2007)

development_windows


score_year,train_start,train_end,score_start,score_end
<dbl>,<date>,<date>,<date>,<date>
2005,2000-01-01,2004-12-31,2005-01-01,2005-12-31
2006,2001-01-01,2005-12-31,2006-01-01,2006-12-31
2007,2002-01-01,2006-12-31,2007-01-01,2007-12-31
2008,2003-01-01,2007-12-31,2008-01-01,2008-12-31
2009,2004-01-01,2008-12-31,2009-01-01,2009-12-31
2010,2005-01-01,2009-12-31,2010-01-01,2010-12-31
2011,2006-01-01,2010-12-31,2011-01-01,2011-12-31
2012,2007-01-01,2011-12-31,2012-01-01,2012-12-31
2013,2008-01-01,2012-12-31,2013-01-01,2013-12-31


## Basic Hyperparameter Tuning

The grid below is intentionally small so it stays easy to run and easy to interpret. Each candidate is evaluated on the same pre-holdout rolling windows, and candidates are ranked by **average monthly rank IC**.


In [ ]:
tuning_grid <- tibble(
  model_id = 1:4,
  size = c(8L, 12L, 16L, 10L),
  decay = c(1e-4, 1e-4, 5e-4, 1e-3),
  maxit = c(150L, 150L, 200L, 150L),
  rang = c(0.05, 0.05, 0.025, 0.025)
)

log_progress("Starting hyperparameter tuning on the selected pre-holdout windows.")

tuning_results <- purrr::pmap_dfr(
  tuning_grid,
  function(model_id, size, decay, maxit, rang) {
    params <- tibble(size = size, decay = decay, maxit = maxit, rang = rang)

    log_progress(
      sprintf(
        "Evaluating tuning candidate %s/4 | size=%s decay=%.4g maxit=%s rang=%.3f",
        model_id,
        size,
        decay,
        maxit,
        rang
      )
    )

    tuning_predictions <- run_walk_forward(
      data_source = training_sample,
      windows = tuning_windows,
      feature_cols = feature_cols,
      target_col = target_col,
      params = params,
      trace = FALSE,
      run_label = sprintf("tuning candidate %s", model_id)
    )

    tuning_summary <- summarise_predictions(tuning_predictions)

    log_progress(
      sprintf(
        paste0(
          "Finished tuning candidate %s | mean monthly rank IC=%.4f | ",
          "RMSE=%.4f | hit ratio=%.4f"
        ),
        model_id,
        tuning_summary$overall$mean_monthly_rank_ic[[1]],
        tuning_summary$overall$rmse[[1]],
        tuning_summary$overall$hit_ratio[[1]]
      )
    )

    tuning_summary$overall %>%
      mutate(
        model_id = model_id,
        size = size,
        decay = decay,
        maxit = maxit,
        rang = rang
      ) %>%
      select(model_id, size, decay, maxit, rang, everything())
  }
) %>%
  arrange(desc(mean_monthly_rank_ic), rmse)

tuning_results


log_progress("Completed hyperparameter tuning.")


[2026-04-12 10:48:15] Starting hyperparameter tuning on the selected pre-holdout windows.
[2026-04-12 10:48:15] Evaluating tuning candidate 1/4 | size=8 decay=0.0001 maxit=150 rang=0.050
[2026-04-12 10:48:15] Running tuning candidate 1 across 3 yearly windows
[2026-04-12 10:48:15] Starting tuning candidate 1 | score_year=2005 | train rows=69,248 (2000-01-31 to 2004-12-31) | score rows=14,332 (2005-01-31 to 2005-12-31) | size=8 decay=0.0001 maxit=150 rang=0.050
[2026-04-12 10:49:35] Finished tuning candidate 1 | score_year=2005
[2026-04-12 10:49:35] Completed tuning candidate 1 | score_year=2005 | rank IC=0.0249 | hit ratio=0.5525
[2026-04-12 10:49:35] Starting tuning candidate 1 | score_year=2006 | train rows=70,510 (2001-01-31 to 2005-12-31) | score rows=14,349 (2006-01-31 to 2006-12-31) | size=8 decay=0.0001 maxit=150 rang=0.050
[2026-04-12 10:50:59] Finished tuning candidate 1 | score_year=2006
[2026-04-12 10:50:59] Completed tuning candidate 1 | score_year=2006 | rank IC=0.0229 | h

In [ ]:
selected_config <- tuning_grid %>%
  filter(model_id == tuning_results$model_id[[1]])

selected_config


## Full Pre-Holdout Walk-Forward Backtest

Once the small tuning exercise picks a configuration, rerun the yearly 5-year rolling backtest across **all available pre-holdout windows**. This gives a cleaner view of how the chosen setup behaved before 2014.


In [ ]:
log_progress("Running the full pre-holdout walk-forward backtest with the selected configuration.")

development_predictions <- run_walk_forward(
  data_source = training_sample,
  windows = development_windows,
  feature_cols = feature_cols,
  target_col = target_col,
  params = selected_config,
  trace = FALSE,
  run_label = "development backtest"
)

development_summary <- summarise_predictions(development_predictions)

log_progress("Completed the full pre-holdout walk-forward backtest.")

development_summary$overall


In [ ]:
ggplot(development_summary$monthly, aes(x = date, y = monthly_rank_ic)) +
  geom_line(color = "steelblue", linewidth = 0.7) +
  geom_hline(yintercept = 0, linetype = "dashed", color = "grey50") +
  labs(
    title = "Pre-holdout Monthly Rank IC",
    x = NULL,
    y = "Rank IC"
  ) +
  theme_minimal(base_size = 12)


## Final Holdout Evaluation

The holdout period is never used for model selection. After tuning is complete, train one final model on the **last 5 pre-holdout years (2009-2013)** and score the untouched holdout sample.


In [ ]:
log_progress("Training the final pre-holdout model and scoring the strict holdout sample.")

final_train_window <- training_sample %>%
  filter(date >= as.Date("2009-01-01"), date <= as.Date("2013-12-31"))

holdout_predictions <- holdout_sample %>%
  mutate(
    predicted_return = fit_and_score_window(
      train_df = final_train_window,
      score_df = holdout_sample,
      feature_cols = feature_cols,
      target_col = target_col,
      size = selected_config$size,
      decay = selected_config$decay,
      maxit = selected_config$maxit,
      rang = selected_config$rang,
      trace = FALSE,
      seed_offset = 999L,
      model_label = "final holdout model"
    )
  ) %>%
  transmute(
    stock_id,
    date,
    actual_return = .data[[target_col]],
    predicted_return
  )

holdout_summary <- summarise_predictions(holdout_predictions)

log_progress("Completed holdout scoring.")

holdout_summary$overall


In [ ]:
ggplot(holdout_summary$monthly, aes(x = date, y = monthly_rank_ic)) +
  geom_line(color = "firebrick", linewidth = 0.7) +
  geom_hline(yintercept = 0, linetype = "dashed", color = "grey50") +
  labs(
    title = "Holdout Monthly Rank IC",
    subtitle = "Model trained on 2009-2013 only",
    x = NULL,
    y = "Rank IC"
  ) +
  theme_minimal(base_size = 12)


## Notes

- Feature scaling is refit separately for every training window and then applied to the corresponding score window.
- The yearly walk-forward loop uses only prior data, so there is no contamination from future returns.
- The holdout set stays untouched until the very end, which makes it suitable for a final out-of-sample check.
- If you want to extend this baseline later, the cleanest next steps are deeper architectures, richer tuning grids, and portfolio-level evaluation metrics.
